### Dataset and Task Metadata

In [31]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="clock_protein_toxicity",
    dataset_year="2021",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://archive.ics.uci.edu/static/public/728/toxicity-2.zip",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder. 

mkdir -p local-data-warehouse/clock_protein_toxicity && wget -P local-data-warehouse/clock_protein_toxicity/ https://archive.ics.uci.edu/static/public/728/toxicity-2.zip && unzip local-data-warehouse/clock_protein_toxicity/toxicity-2.zip -d local-data-warehouse/clock_protein_toxicity/ && rm local-data-warehouse/clock_protein_toxicity/toxicity-2.zip
""",
    # References
    academic_reference_bibtex="""@article{Gul2021StructureBasedDC,
  title={Structure-based design and classifications of small molecules regulating the circadian rhythm period},
  author={Sadia Gul and Fatima Rahim and Selen Isin and others},
  journal={Scientific Reports},
  year={2021},
  volume={11},
  pages={18510},
  doi={10.1038/s41598-021-97962-5},
  url={https://doi.org/10.1038/s41598-021-97962-5}
}
""",
    academic_reference_bibtex_key="Gul2021StructureBasedDC",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
    - We remove duplicated columns but keep the first occurance.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Toxic",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="Toxic",
)

## Preprocessing

In [32]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "data.csv")

target_feature = "Toxic"
df = df.rename(columns={"Class": target_feature})
df[target_feature] = df[target_feature].map({"Toxic": "Yes", "NonToxic": "No"}).astype("category")

duplicated_mask = df.T.duplicated(keep='first')
df = df.drop(columns = df.columns[duplicated_mask])

print("Loaded data shape:", df.shape)

Loaded data shape: (171, 1112)


In [33]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,MATS3v,nHBint10,MATS3s,MATS3p,nHBDon_Lipinski,minHBint8,MATS3e,MATS3c,minHBint2,MATS3m,minHBint6,minHBint7,minHBint4,MATS3i,VR3_Dt,SpMax8_Bhi,SdsN,SpMax8_Bhm,SpMax8_Bhe,ECCEN,MDEC-14,SpMax8_Bhs,SpMax8_Bhp,SpMax8_Bhv,MDEC-11,MDEC-12,MDEC-13,VR2_Dt,BIC5,ATS7s,ATS7p,ATS7v,ATS7i,ATS7m,ATS7e,mintN,nHsNH2,khs.sssCH,minHBint3,maxdssC,nT6Ring,minHBint5,nF8Ring,minssCH2,SpMax_DzZ,ETA_EtaP,nHsOH,SpMin1_Bhe,maxHother,nHBAcc_Lipinski,StN,khs.aaS,khs.aaO,khs.aaN,Sare,SHAvin,SpMax3_Bhv,SpMax3_Bhp,SpMax3_Bhs,SpMax3_Bhe,SpMin6_Bhi,SpMax3_Bhm,SpMax3_Bhi,ETA_EtaP_F_L,mindCH2,AATSC2e,AATSC2c,AATSC2m,AATSC2i,nsBr,AATS5p,AATSC2v,AATSC2p,AATSC2s,VABC,maxdNH,khs.ddsN,RotBtFrac,ATS4e,ATS4m,nFRing,ATS4i,ATS4s,ATS4p,ETA_Alpha,khs.sssN,EE_Dzi,MAXDN,EE_Dzm,EE_Dze,EE_Dzs,EE_Dzp,EE_Dzv,ATS8e,maxsOH,minssssNp,maxsOm,MDEC-23,MDEC-22,MDEC-24,nFG12HeteroRing,ATS8s,ATS8v,SP-6,SP-7,SHsNH2,SP-5,SP-2,SP-3,SP-0,SP-1,minHsOH,ATSC8v,MATS2v,ATSC8s,MATS2p,MATS2s,ATSC8p,MATS2e,ATSC8e,ATSC8c,MATS2c,MATS2m,topoDiameter,ATSC8m,MATS2i,ATSC8i,ntN,khs.ssCH2,SpAD_Dt,ETA_Eta_R_L,SHdsCH,SaasN,SC-4,SaasC,minaaCH,AATSC3c,AATSC3e,AATSC3i,AATSC3m,AATSC3s,AATSC3p,AATSC3v,SpMax2_Bhp,AATS8e,AATS8i,AATS8m,AATS8s,AATS8p,AATS8v,VE3_Dt,XLogP,SpMax2_Bhi,maxssCH2,minaaS,SpMax4_Bhv,SpMax4_Bhs,SpMax4_Bhp,SpMax4_Bhm,minHaaNH,SpMax4_Bhi,minaaN,SpMax4_Bhe,StsC,SssCH2,maxHdNH,MATS1p,R_TpiPCTPC,MATS1s,MATS1v,JGI10,MATS1c,MATS1e,VR2_DzZ,MATS1i,MATS1m,MDEC-34,MDEC-33,VR2_Dze,VR2_Dzm,VR2_Dzs,VR2_Dzp,khs.ssssC,nTG12Ring,khs.ssssN,ATS5e,gmin,VR2_D,ATS5m,ATS5p,ATS5s,ATS5v,AATSC5p,TpiPC,maxsCH3,SdS,khs.ssO,ETA_Eta_F_L,khs.ssS,SdO,VE2_Dt,maxHtCH,ETA_dEpsilon_B,ETA_dEpsilon_A,ETA_dEpsilon_D,SsNH2,StCH,SsCH3,CIC5,CIC4,CIC1,CIC0,CIC3,CIC2,nF10HeteroRing,maxssO,WPOL,n5HeteroRing,maxHAvin,fragC,ETA_Eta_B_RC,AATS7m,SpDiam_Dt,SdssC,ETA_Epsilon_3,AATS7i,AATS7e,nT9Ring,minsCl,AATS7v,AATS7s,AATS7p,nHdCH2,ETA_Epsilon_5,ETA_Epsilon_4,SsssCH,maxHsOH,GATS1v,maxaaaC,GATS1s,minsNH2,BIC4,SpMin7_Bhs,SpMin7_Bhp,SpMin7_Bhv,nHtCH,GATS1e,mintsC,GATS1c,SpMin7_Bhm,GATS1m,SpMin7_Bhe,GATS1i,maxtsC,minHAvin,MDEC-44,AATS2v,SPC-6,SPC-4,SPC-5,SpAD_D,MATS6c,ETA_BetaP_s,minaasC,minaasN,minssNH,nT7HeteroRing,RotBFrac,nF10Ring,ETA_BetaP_ns,nH,nL,nN,nO,nA,nC,nF,nX,nQ,nS,nV,ATS1m,mindsN,SHCsats,SHCsatu,CrippenMR,GATS1p,SRW10,ETA_dPsi_A,AATS6m,AATS6i,AATS6e,minsBr,nF9HeteroRing,SpMin7_Bhi,AATS6v,AATS6p,AATS6s,naAromAtom,nBase,minHBint10,SpDiam_DzZ,SaaNH,khs.dNH,maxaaN,maxaaO,SpDiam_Dzv,SpDiam_Dzs,SpDiam_Dzp,SpDiam_Dze,SpDiam_Dzm,GATS6i,SpDiam_Dzi,Mi,Mv,Mp,GGI10,bpol,MW,GATS6v,MATS7s,MATS7p,C1SP1,MATS7v,MATS7i,MATS7m,MATS7c,GATS6s,MATS7e,maxtN,SpMin8_Bhe,SpMin8_Bhi,SpMin8_Bhm,SpMin8_Bhp,SpMin8_Bhs,SpMin8_Bhv,maxHCsatu,maxHCsats,ATSC3v,ATSC3s,ATSC3p,minHdCH2,ATSC3e,ATSC3c,maxssNH,ATSC3m,ATSC3i,minHCsatu,minHCsats,SpMax7_Bhe,SpMax7_Bhi,SpMax7_Bhm,ETA_BetaP_ns_d,SpMax7_Bhp,SpMax7_Bhs,SpMax7_Bhv,SdsCH,minssO,minssS,SpMin3_Bhe,SpMin3_Bhm,SpMin3_Bhi,SpMin3_Bhv,SpMin3_Bhs,SpMin3_Bhp,TPC,VP-5,VP-4,VP-7,VP-6,VP-1,VP-0,VP-3,VP-2,MIC5,MIC4,MIC3,MIC2,MIC1,MIC0,ATSC5p,piPC10,ATSC5s,minsOm,nHBa,nHBd,SddssS,nCl,minsOH,SHaaCH,nHBDon,nF11HeteroRing,AATS5i,AATS5m,SpMin6_Bhs,SpMin6_Bhp,SpMin6_Bhv,AATS5e,ETA_dBeta,khs.sCH3,ALogP,SpMin6_Bhm,BCUTp-1l,AATS5s,BCUTp-1h,AATS5v,SpMin6_Bhe,ATS8i,ATS8m,BCUTw-1h,BCUTw-1l,nBondsS3,nBondsS2,ATS8p,GATS3p,GATS3s,GATS3v,GATS3c,GATS3e,SC-5,GATS3i,SC-6,GATS3m,SC-3,minsCH3,SssssC,nAtomLC,nT12HeteroRing,minHaaCH,MLFER_BH,MLFER_BO,SaaaC,mindsCH,nddssS,maxaasC,maxsssN,MATS6p,MATS6s,MATS6v,MATS6i,MATS6m,MATS6e,ETA_Beta_ns_d,hmax,ETA_Beta_s,nHaaCH,khs.aaaC,ETA_AlphaP,nAromBond,ATSC2v,ATSC2p,ATSC2s,ATS4v,ATSC2e,ATSC2c,ATSC2m,ATSC2i,AATS4v,AATS4s,AATS4p,AATS4e,ETA_BetaP,AATS4m,AATS4i,SHsOH,SpMax_D,MDEN-13,MDEN-12,MDEN-11,ATS3m,PetitjeanNumber,khs.aasN,khs.aasC,MWC10,MPC7,TWC,topoRadius,WPATH,nG,ndsN,MAXDP,naaaC,SM1_DzZ,SpAbs_DzZ,SpAbs_Dze,khs.aaNH,SpAbs_Dzm,SM1_Dzv,SpAbs_Dzi,SM1_Dzp,SM1_Dzs,SM1_Dzm,SpAbs_Dzv,SM1_Dzi,SpAbs_Dzp,SpAbs_Dzs,SM1_Dze,maxaaCH,maxdO,nF11Ring,GATS2v,GATS2s,ETA_Beta,GATS2m,GATS2i,GATS2e,GATS2c,nHaaNH,ATSC1p,ATSC1s

## Data Checks

In [34]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 171
Columns: 1112
Use sampling: False (sample size: 171)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['SpMAD_Dzs', 'GATS6m', 'ATS5v', 'MATS8s', 'AATSC6m', 'AATSC6v', 'AATS7i', 'AATS7v', 'AATS7p', 'ATSC5i']
Rows remaining as candidates after top-10 filter: 0 (of 171)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [35]:
# Sample Rows
df_head

,MATS3v,nHBint10,MATS3s,MATS3p,nHBDon_Lipinski,minHBint8,MATS3e,MATS3c,minHBint2,MATS3m,minHBint6,minHBint7,minHBint4,MATS3i,VR3_Dt,SpMax8_Bhi,SdsN,SpMax8_Bhm,SpMax8_Bhe,ECCEN,MDEC-14,SpMax8_Bhs,SpMax8_Bhp,SpMax8_Bhv,MDEC-11,MDEC-12,MDEC-13,VR2_Dt,BIC5,ATS7s,ATS7p,ATS7v,ATS7i,ATS7m,ATS7e,mintN,nHsNH2,khs.sssCH,minHBint3,maxdssC,nT6Ring,minHBint5,nF8Ring,minssCH2,SpMax_DzZ,ETA_EtaP,nHsOH,SpMin1_Bhe,maxHother,nHBAcc_Lipinski,StN,khs.aaS,khs.aaO,khs.aaN,Sare,SHAvin,SpMax3_Bhv,SpMax3_Bhp,SpMax3_Bhs,SpMax3_Bhe,SpMin6_Bhi,SpMax3_Bhm,SpMax3_Bhi,ETA_EtaP_F_L,mindCH2,AATSC2e,AATSC2c,AATSC2m,AATSC2i,nsBr,AATS5p,AATSC2v,AATSC2p,AATSC2s,VABC,maxdNH,khs.ddsN,RotBtFrac,ATS4e,ATS4m,nFRing,ATS4i,ATS4s,ATS4p,ETA_Alpha,khs.sssN,EE_Dzi,MAXDN,EE_Dzm,EE_Dze,EE_Dzs,EE_Dzp,EE_Dzv,ATS8e,maxsOH,minssssNp,maxsOm,MDEC-23,MDEC-22,MDEC-24,nFG12HeteroRing,ATS8s,ATS8v,SP-6,SP-7,SHsNH2,SP-5,SP-2,SP-3,SP-0,SP-1,minHsOH,ATSC8v,MATS2v,ATSC8s,MATS2p,MATS2s,ATSC8p,MATS2e,ATSC8e,ATSC8c,MATS2c,MATS2m,topoDiameter,ATSC8m,MATS2i,ATSC8i,ntN,khs.ssCH2,SpAD_Dt,ETA_Eta_R_L,SHdsCH,SaasN,SC-4,SaasC,minaaCH,AATSC3c,AATSC3e,AATSC3i,AATSC3m,AATSC3s,AATSC3p,AATSC3v,SpMax2_Bhp,AATS8e,AATS8i,AATS8m,AATS8s,AATS8p,AATS8v,VE3_Dt,XLogP,SpMax2_Bhi,maxssCH2,minaaS,SpMax4_Bhv,SpMax4_Bhs,SpMax4_Bhp,SpMax4_Bhm,minHaaNH,SpMax4_Bhi,minaaN,SpMax4_Bhe,StsC,SssCH2,maxHdNH,MATS1p,R_TpiPCTPC,MATS1s,MATS1v,JGI10,MATS1c,MATS1e,VR2_DzZ,MATS1i,MATS1m,MDEC-34,MDEC-33,VR2_Dze,VR2_Dzm,VR2_Dzs,VR2_Dzp,khs.ssssC,nTG12Ring,khs.ssssN,ATS5e,gmin,VR2_D,ATS5m,ATS5p,ATS5s,ATS5v,AATSC5p,TpiPC,maxsCH3,SdS,khs.ssO,ETA_Eta_F_L,khs.ssS,SdO,VE2_Dt,maxHtCH,ETA_dEpsilon_B,ETA_dEpsilon_A,ETA_dEpsilon_D,SsNH2,StCH,SsCH3,CIC5,CIC4,CIC1,CIC0,CIC3,CIC2,nF10HeteroRing,maxssO,WPOL,n5HeteroRing,maxHAvin,fragC,ETA_Eta_B_RC,AATS7m,SpDiam_Dt,SdssC,ETA_Epsilon_3,AATS7i,AATS7e,nT9Ring,minsCl,AATS7v,AATS7s,AATS7p,nHdCH2,ETA_Epsilon_5,ETA_Epsilon_4,SsssCH,maxHsOH,GATS1v,maxaaaC,GATS1s,minsNH2,BIC4,SpMin7_Bhs,SpMin7_Bhp,SpMin7_Bhv,nHtCH,GATS1e,mintsC,GATS1c,SpMin7_Bhm,GATS1m,SpMin7_Bhe,GATS1i,maxtsC,minHAvin,MDEC-44,AATS2v,SPC-6,SPC-4,SPC-5,SpAD_D,MATS6c,ETA_BetaP_s,minaasC,minaasN,minssNH,nT7HeteroRing,RotBFrac,nF10Ring,ETA_BetaP_ns,nH,nL,nN,nO,nA,nC,nF,nX,nQ,nS,nV,ATS1m,mindsN,SHCsats,SHCsatu,CrippenMR,GATS1p,SRW10,ETA_dPsi_A,AATS6m,AATS6i,AATS6e,minsBr,nF9HeteroRing,SpMin7_Bhi,AATS6v,AATS6p,AATS6s,naAromAtom,nBase,minHBint10,SpDiam_DzZ,SaaNH,khs.dNH,maxaaN,maxaaO,SpDiam_Dzv,SpDiam_Dzs,SpDiam_Dzp,SpDiam_Dze,SpDiam_Dzm,GATS6i,SpDiam_Dzi,Mi,Mv,Mp,GGI10,bpol,MW,GATS6v,MATS7s,MATS7p,C1SP1,MATS7v,MATS7i,MATS7m,MATS7c,GATS6s,MATS7e,maxtN,SpMin8_Bhe,SpMin8_Bhi,SpMin8_Bhm,SpMin8_Bhp,SpMin8_Bhs,SpMin8_Bhv,maxHCsatu,maxHCsats,ATSC3v,ATSC3s,ATSC3p,minHdCH2,ATSC3e,ATSC3c,maxssNH,ATSC3m,ATSC3i,minHCsatu,minHCsats,SpMax7_Bhe,SpMax7_Bhi,SpMax7_Bhm,ETA_BetaP_ns_d,SpMax7_Bhp,SpMax7_Bhs,SpMax7_Bhv,SdsCH,minssO,minssS,SpMin3_Bhe,SpMin3_Bhm,SpMin3_Bhi,SpMin3_Bhv,SpMin3_Bhs,SpMin3_Bhp,TPC,VP-5,VP-4,VP-7,VP-6,VP-1,VP-0,VP-3,VP-2,MIC5,MIC4,MIC3,MIC2,MIC1,MIC0,ATSC5p,piPC10,ATSC5s,minsOm,nHBa,nHBd,SddssS,nCl,minsOH,SHaaCH,nHBDon,nF11HeteroRing,AATS5i,AATS5m,SpMin6_Bhs,SpMin6_Bhp,SpMin6_Bhv,AATS5e,ETA_dBeta,khs.sCH3,ALogP,SpMin6_Bhm,BCUTp-1l,AATS5s,BCUTp-1h,AATS5v,SpMin6_Bhe,ATS8i,ATS8m,BCUTw-1h,BCUTw-1l,nBondsS3,nBondsS2,ATS8p,GATS3p,GATS3s,GATS3v,GATS3c,GATS3e,SC-5,GATS3i,SC-6,GATS3m,SC-3,minsCH3,SssssC,nAtomLC,nT12HeteroRing,minHaaCH,MLFER_BH,MLFER_BO,SaaaC,mindsCH,nddssS,maxaasC,maxsssN,MATS6p,MATS6s,MATS6v,MATS6i,MATS6m,MATS6e,ETA_Beta_ns_d,hmax,ETA_Beta_s,nHaaCH,khs.aaaC,ETA_AlphaP,nAromBond,ATSC2v,ATSC2p,ATSC2s,ATS4v,ATSC2e,ATSC2c,ATSC2m,ATSC2i,AATS4v,AATS4s,AATS4p,AATS4e,ETA_BetaP,AATS4m,AATS4i,SHsOH,SpMax_D,MDEN-13,MDEN-12,MDEN-11,ATS3m,PetitjeanNumber,khs.aasN,khs.aasC,MWC10,MPC7,TWC,topoRadius,WPATH,nG,ndsN,MAXDP,naaaC,SM1_DzZ,SpAbs_DzZ,SpAbs_Dze,khs.aaNH,SpAbs_Dzm,SM1_Dzv,SpAbs_Dzi,SM1_Dzp,SM1_Dzs,SM1_Dzm,SpAbs_Dzv,SM1_Dzi,SpAbs_Dzp,SpAbs_Dzs,SM1_Dze,maxaaCH,maxdO,nF11Ring,GATS2v,GATS2s,ETA_Beta,GATS2m,GATS2i,GATS2e,GATS2c,nHaaNH,ATSC1p,ATSC1s

In [36]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Toxic,category,0.0,0.0,2.0,"No, Yes"
1,MATS3v,float64,0.0,0.0,163.0,"0.0096, -0.0619, 0.03, 0.0289, 0.0247, -0.0325, -0.0028, -0.0251, -0.0702, -0.0432"
2,MATS3s,float64,0.0,0.0,168.0,"0.017, -0.0384, 0.0095, 0.0065, -0.0678, -0.0022, 0.0489, -0.0116, -0.0282, -0.0319"
3,MATS3p,float64,0.0,0.0,157.0,"0.0089, -0.0137, -0.0905, -0.1369, -0.0614, -0.0429, -0.0164, -0.0453, -0.1607, -0.0912"
4,minHBint8,float64,0.0,0.0,38.0,"0.0, 4.3121, 2.1399, 4.0632, 0.7339, 5.385, 0.9326, 0.6583, 0.9507, 0.5115"
5,MATS3e,float64,0.0,0.0,162.0,"-0.0436, -0.0332, -0.0201, -0.0739, -0.0265, -0.0549, -0.0663, -0.0763, -0.0356, -0.0185"
6,MATS3c,float64,0.0,0.0,169.0,"-0.045, -0.0876, 0.0409, -0.2554, 0.0461, -0.0257, 0.1063, 0.0762, -0.1663, -0.118"
7,minHBint2,float64,0.0,0.0,67.0,"0.0, 5.0375, 5.096, -0.663, 0.6969, 4.9615, 5.4463, 5.4595, 5.8704, 5.9765"
8,MATS3m,float64,0.0,0.0,163.0,"0.0143, 0.0511, 0.0361, -0.0669, 0.0922, 0.1063, -0.0337, 0.0086, -0.0417, 0.0471"
9,minHBint6,float64,0.0,0.0,51.0,"0.0, 4.5206, 4.4584, 0.289, 0.571, 1.001, 0.5781, 0.5757, 0.3674, 3.3506"


In [37]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
MATS3v,171.0,-0.031244,6.355880e-02,-0.3115,1.411000e-01
nHBint10,171.0,0.315789,7.629177e-01,0.0000,4.000000e+00
MATS3s,171.0,-0.001001,6.392783e-02,-0.1846,2.181000e-01
MATS3p,171.0,-0.061501,7.289129e-02,-0.3485,1.290000e-01
nHBDon_Lipinski,171.0,0.994152,1.108773e+00,0.0000,6.000000e+00
minHBint8,171.0,0.677770,1.647322e+00,0.0000,8.141400e+00
MATS3e,171.0,-0.025418,7.864469e-02,-0.2119,2.495000e-01
MATS3c,171.0,-0.053289,1.094628e-01,-0.4729,2.122000e-01
minHBint2,171.0,1.569251,2.497362e+00,-0.7087,7.740800e+00
MATS3m,171.0,0.003226,7.407627e-02,-0.1987,1.684000e-01


In [38]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column rank                    
Toxic  1       No    115  67.25
       2      Yes     56  32.75

In [39]:
# Target Distribution
target_df

,count,pct
Toxic,,
No,115,67.25
Yes,56,32.75


## Task Curation

In [40]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [41]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [42]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019cf608-902b-7180-a7d2-f5944d5c8f70
107c9cfbcf26d9a413d9653913711d855d7ed72ee1fa6dc8ae150ed7124dd89c
